# LAB 08 - Pretrained Transformers

Giống notebook Lab7: **Run All** để train lại, in tiến độ và lưu checkpoint/dữ liệu trung gian vào `training_output/`, xuất CSV/PNG ra cùng thư mục với code. Không nhúng kết quả cũ.

Cùng tập IMDb 4.000 mẫu, split 2.800/400/800 và seed 42 như Lab7. Dùng BERT nhỏ 2 layers, 128 hidden, 2 heads để giảm chi phí. Ba chế độ dùng cùng kiến trúc/tokenizer; `scratch` là BERT khởi tạo ngẫu nhiên, không phải checkpoint Transformer Lab7.

Cài trong kernel đang chạy: `%pip install torch "transformers>=4.40,<5" numpy pandas scikit-learn matplotlib ipykernel`. Muốn dùng NVIDIA GPU cần PyTorch hỗ trợ CUDA.

In [1]:
from pathlib import Path
from pretrained_classification import run, MODEL

DATASET_PATH = Path('datasets/imdb/IMDB Dataset.csv').resolve()
OUTPUT_DIR = Path('training_output').resolve()
EPOCHS = 4
DEVICE = 'auto'
MAX_LENGTH = 64
BATCH_SIZE = 32
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'Sửa DATASET_PATH: {DATASET_PATH}')

C:\Users\truon\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Pretrained vs from scratch; frozen vs fine-tuning

`[CLS]` representation đi qua dropout và linear layer để tạo 2 logits. Cross-entropy nhận logits; softmax dùng khi đánh giá. Frozen encoder tắt gradient và dropout của encoder, chỉ train head. Chọn checkpoint theo validation loss, sau đó mới đánh giá test. Learning rate: scratch/frozen `1e-3`, fine-tuning `2e-5`; vì vậy đây là so sánh các chế độ có cấu hình học phù hợp, không cô lập duy nhất yếu tố pretrained weights.

In [2]:
results = run(
    data_path=DATASET_PATH, output_dir=OUTPUT_DIR,
    epochs=EPOCHS, device=DEVICE, model_name=MODEL,
    modes=('scratch', 'frozen', 'finetuned'),
    max_length=MAX_LENGTH, batch_size=BATCH_SIZE,
)

scratch data=100% 1/4: train=0.6886, validation=0.5475


scratch data=100% 2/4: train=0.3413, validation=0.5821


scratch data=100% 3/4: train=0.1048, validation=0.9396


scratch data=100% 4/4: train=0.0362, validation=1.3915


frozen data=100% 1/4: train=0.6757, validation=0.6424


frozen data=100% 2/4: train=0.6355, validation=0.6129


... (2 outputs left out)

finetuned data=100% 1/4: train=0.7108, validation=0.6651


finetuned data=100% 2/4: train=0.6728, validation=0.6450


finetuned data=100% 3/4: train=0.6406, validation=0.6214


finetuned data=100% 4/4: train=0.6197, validation=0.6007


## Kết quả

`results.csv` cạnh code lưu accuracy, macro precision/recall/F1, thời gian và số tham số. `confusion_matrix.png` được tạo sau training. File predictions lưu nhãn thật, nhãn dự đoán, xác suất positive và confidence. Mọi số liệu được tính từ lần chạy thực tế.

## Amount of training data

Train lại từ pretrained checkpoint cho từng mức 10%, 25%, 50%, 100% (280/700/1400/2800 mẫu), 4 epoch, seed 42, learning rate như nhau. Các subset lồng nhau từ cùng permutation; validation/test giữ nguyên. Chọn checkpoint theo validation loss, chỉ dùng test để báo cáo. Đây là một seed và một lịch huấn luyện cố định; số optimizer steps tăng theo lượng dữ liệu.

In [3]:
from pretrained_classification import run_data_efficiency
data_results = run_data_efficiency(
    data_path=DATASET_PATH, output_dir=OUTPUT_DIR,
    epochs=EPOCHS, device=DEVICE, model_name=MODEL,
    max_length=MAX_LENGTH, batch_size=BATCH_SIZE,
)

finetuned data=10% 1/4: train=0.7437, validation=0.7032


finetuned data=10% 2/4: train=0.7177, validation=0.6992


finetuned data=10% 3/4: train=0.6881, validation=0.6925


finetuned data=10% 4/4: train=0.7074, validation=0.6812


finetuned data=25% 1/4: train=0.7376, validation=0.6960


finetuned data=25% 2/4: train=0.7082, validation=0.6837


... (6 outputs left out)

finetuned data=100% 1/4: train=0.7108, validation=0.6651


finetuned data=100% 2/4: train=0.6728, validation=0.6450


finetuned data=100% 3/4: train=0.6406, validation=0.6214


finetuned data=100% 4/4: train=0.6197, validation=0.6007


## Confidence analysis và file kết quả

`confidence_analysis.png` được tính từ softmax của 800 dự đoán test của fine-tuned model ở thí nghiệm chính. Confidence là xác suất lớn nhất; nhóm Low: [0.5, 0.7], Medium: (0.7, 0.9], High: (0.9, 1.0]. Nhóm rỗng ghi rõ số mẫu bằng 0.

`results.csv`, `data_efficiency.png`, `confusion_matrix.png`, `confidence_analysis.png` tự động được xuất cạnh file Python. Dữ liệu gốc của biểu đồ, split, cấu hình và checkpoint vẫn lưu trong `training_output/`. Error analysis và reflection chưa làm.